# 01 tokenizer + model

目标：拆开 `AutoTokenizer` 和 `AutoModelForCausalLM`，看清楚文本如何变成 token ids，又如何 decode 回文本。


## 运行环境准备

这个 notebook 默认使用 `Qwen/Qwen2.5-0.5B-Instruct`，适合在魔搭 Notebook 里快速学习。

如果你想用更大的模型，可以把 `MODEL_ID` 改成 `Qwen/Qwen2.5-7B-Instruct`，然后重启内核重新运行。


In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}


In [ ]:
import os
from pathlib import Path

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE != "modelscope":
        return model_id

    from modelscope import snapshot_download
    return snapshot_download(model_id)


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)


## 1. 加载 tokenizer 和 model

这一层开始脱离 `pipeline`，显式控制模型和分词器。


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


## 2. messages 到 prompt

`chat_template` 负责把 system/user/assistant 消息渲染成模型训练时熟悉的格式。


In [ ]:
messages = [
    {"role": "system", "content": "你是一个讲解大模型部署的老师。"},
    {"role": "user", "content": "用三句话解释什么是 chat template。"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(prompt)


## 3. tokenize、generate、decode

`generate()` 输出包含输入 token 和新生成 token，所以 decode 时通常只取新增部分。


In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=False,
)

new_token_ids = outputs[0][inputs["input_ids"].shape[-1] :]
answer = tokenizer.decode(new_token_ids, skip_special_tokens=True)

print(answer)
